# 03 - Model Experiments

This notebook experiments with different model architectures and hyperparameters.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.models.train_model import ModelTrainer
from src.models.predict_model import Predictor
from src.models.evaluate_model import ModelEvaluator
from src.utils.metrics import Metrics

print('Model modules imported successfully')

## Generate Synthetic Data

In [ ]:
# Generate synthetic training data
np.random.seed(42)

n_samples = 200
n_features = 50
n_outputs = 5  # OCEAN traits

X = np.random.randn(n_samples, n_features)
y = np.random.rand(n_samples, n_outputs)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set shape: {X_train_scaled.shape}')
print(f'Test set shape: {X_test_scaled.shape}')

## Train Random Forest Model

In [ ]:
# Train random forest model
trainer = ModelTrainer(model_type='random_forest')
trainer.fit(X_train_scaled, y_train)

# Make predictions
y_pred = trainer.predict(X_test_scaled)

print(f'Predictions shape: {y_pred.shape}')
print(f'Sample predictions:\n{y_pred[:5]}')

## Model Evaluation

In [ ]:
# Evaluate model
traits = ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']
per_output_metrics = ModelEvaluator.evaluate_per_output(y_test, y_pred, output_names=traits)

# Display results
print('Per-trait evaluation:')
for trait, metrics in per_output_metrics.items():
    print(f'\n{trait}:')
    for metric_name, value in metrics.items():
        print(f'  {metric_name}: {value:.4f}')

## Compare with Ridge Model

In [ ]:
# Train ridge model
ridge_trainer = ModelTrainer(model_type='ridge')
ridge_trainer.fit(X_train_scaled, y_train)

# Make predictions
ridge_pred = ridge_trainer.predict(X_test_scaled)

# Compare metrics
print('Random Forest R² score:', ModelEvaluator.evaluate(y_test, y_pred)['r2_score'])
print('Ridge R² score:', ModelEvaluator.evaluate(y_test, ridge_pred)['r2_score'])